# PyTorch Fundamentals — The Definitive Reference
**HackerRank Prep — Theme 0.6**

## TL;DR
PyTorch is the backbone of virtually every modern deep learning model used in structural biology, protein language models, and ML research. This notebook is the **shared foundation** for:
- Notebook 05 (Deep Learning)
- Notebook 06 (Graph Neural Networks)
- Notebook 07 (AlphaFold3 & protein structure)
- Notebook 10 (Fine-tuning transformers — OpenFOLD3 capstone)

Every concept introduced here will be used directly in those notebooks. Master this first.

---

## Learning Objectives
- [ ] Create, manipulate, and move tensors (CPU ↔ GPU) fluently
- [ ] Understand autograd: computational graph, `.backward()`, `.grad`
- [ ] Build custom `nn.Module` networks from scratch
- [ ] Write a complete, correct training loop (DataLoader → optimizer → loss → backward → step)
- [ ] Save and load model checkpoints; use device-agnostic code
- [ ] Use hooks, `named_parameters()`, and mixed precision (`torch.autocast`)
- [ ] Implement `nn.Embedding`, `nn.MultiheadAttention`, and a simple transformer block
- [ ] Debug shape mismatches, NaN gradients, and in-place operation errors

---

## Self-Check After Completing This Notebook
1. From memory: write a 3-layer MLP `nn.Module` with ReLU activations.
2. What is the difference between `tensor.detach()` and `tensor.data`? Why does it matter?
3. What does `.zero_grad()` do and why must it be called before `.backward()`?
4. How do you make code that runs on CPU and GPU without if/else branches everywhere?
5. Explain in one sentence what `torch.autocast` does and why it speeds up training.

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with little or no ML background.

**How to study it on a first pass:**
- read every markdown section before touching the code
- run the code in small chunks
- paraphrase each concept in your own words
- write one tiny example from memory after finishing

**Common traps:** memorizing syntax without understanding the data flow, skipping probability, and rushing past train/test split discipline.


## Canonical Resource Upgrade

If you need a second explanation, use these first:
- [CS50P](https://cs50.harvard.edu/python/) for programming fundamentals
- [Harvard Stat 110](https://stat110.hsites.harvard.edu/) for probability intuition
- [scikit-learn MOOC](https://inria.github.io/scikit-learn-mooc/) for correct ML workflow
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) once you reach the deep learning notebooks


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 00/01 Python Core + 00/02 ML Fundamentals — numpy, calculus intuition, gradient descent concept |
| You Are Here | Module 00/06 — PyTorch Fundamentals |
| Next Steps | 05/01 Deep Learning & Fine-tuning, 06/01 Structural ML/GNNs, 07/01 AlphaFold3, 10/01 OpenFold3 Fine-tuning |
| Goal | Implement full PyTorch training loop; understand autograd; run ESM2 protein inference |

### From Scratch? Start Here:
1. [Karpathy Neural Networks Zero to Hero (free)](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) — build backprop from scratch
2. [PyTorch 60-minute blitz (free)](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html)
3. This notebook

**Time:** 12-20 hours | **Difficulty:** ⭐⭐⭐⭐

### Cross-References
- Builds on: 00/01 Python Core, 00/02 ML Fundamentals
- Used by: 05/01 Deep Learning, 06/01 Structural ML, 07/01-07/04 AlphaFold3, 10/01 OpenFold3

## 🤖 AI Walkthrough — Claude Prompts

Open a Claude conversation alongside this notebook and use these prompts:

**On tensors and autograd:**
```
Explain PyTorch's computational graph to me.
When I call loss.backward(), what is stored in the graph and how does PyTorch
know how to compute each gradient? Walk through a simple y = x*w + b example
step by step, showing what leaf vs non-leaf tensors are.
```

**On nn.Module:**
```
Explain how nn.Module works in PyTorch.
What does __init__ vs forward do? Where are parameters stored?
If I call model.parameters(), what exactly does it return and why does
the optimizer need it?
```

**On the training loop:**
```
Walk me through the exact sequence of a PyTorch training loop step:
1. zero_grad()
2. forward pass
3. loss computation
4. backward()
5. optimizer.step()
For each step: what state changes? What would break if I skipped it?
```

**On device management:**
```
Explain PyTorch device management. What happens when a tensor is on CPU
but the model is on GPU? Show me the idiomatic pattern for writing device-agnostic
code that works whether CUDA is available or not.
```

**On attention mechanisms:**
```
Explain nn.MultiheadAttention in PyTorch.
What are the q, k, v inputs? What does 'multihead' add vs single-head attention?
Why does AlphaFold and ESM2 use this as their core building block?
```

**On debugging:**
```
I'm getting 'RuntimeError: Expected all tensors to be on the same device'.
Walk me through the systematic debugging process. What are the 5 most common
causes of this error and how do I fix each one?
```

---

## Interview Tip Bank
> **Most common PyTorch interview question**: Explain the training loop. They want to hear: zero_grad → forward → loss → backward → step, and WHY each step is needed.

> **Autograd gotcha**: Gradients accumulate by default. Forgetting `.zero_grad()` is the #1 beginner bug — gradients from previous batches pile up.

> **eval() vs train()**: Always call `model.eval()` before inference and `model.train()` before training. BatchNorm and Dropout behave differently in each mode.

> **Leaf vs non-leaf tensors**: Only leaf tensors (inputs, parameters) with `requires_grad=True` accumulate `.grad`. Intermediate tensors store only the gradient function.

> **Mixed precision**: `torch.autocast` uses float16 for most ops, float32 for numerically sensitive ones. ~2x speedup, ~2x memory reduction on modern GPUs.

## Prerequisites

Before this notebook, you should be comfortable with:
- **Notebook 00/01**: Python basics (classes, functions, list comprehensions)
- **Notebook 00/01**: NumPy arrays (shapes, broadcasting, indexing)
- **Notebook 02**: Basic ML concepts (loss functions, gradient descent conceptually)

You do NOT need prior PyTorch experience. Everything is built from the ground up.

### Quick NumPy-to-PyTorch mapping (reference)
| NumPy | PyTorch | Notes |
|-------|---------|-------|
| `np.array([1,2,3])` | `torch.tensor([1,2,3])` | Creates from data |
| `np.zeros((3,4))` | `torch.zeros(3,4)` | Zeros tensor |
| `arr.shape` | `tensor.shape` | Same API |
| `arr @ arr2` | `tensor @ tensor2` | Matrix multiply |
| `arr.T` | `tensor.T` | Transpose |
| `arr.reshape(2,-1)` | `tensor.reshape(2,-1)` | Reshape |
| `arr.astype(np.float32)` | `tensor.float()` | Type cast |
| `tensor.numpy()` | `arr` | Back to NumPy (CPU only) |

### Install check
```bash
pip install torch torchvision  # CPU-only
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121  # CUDA 12.1
```

## Section 1 — Tensors & Autograd

### What is a Tensor?
A tensor is a multi-dimensional array — PyTorch's equivalent of NumPy's `ndarray`. The critical difference: tensors can **track gradient computation** and live on a **GPU**.

### What is Autograd?
Autograd is PyTorch's automatic differentiation engine. Every operation on a tensor with `requires_grad=True` is recorded in a **computation graph**. When you call `.backward()`, PyTorch traverses this graph in reverse (backpropagation) and populates `.grad` on all leaf tensors.

```
Forward pass:    x  →  w*x  →  loss
Backward pass:   dloss/dw  ←  (chain rule applied at each node)
```

### Key Concepts
- **Leaf tensor**: Created directly by user (model weights, inputs). These accumulate `.grad`.
- **Non-leaf tensor**: Result of an operation. Stores only the gradient function (`.grad_fn`).
- **`.detach()`**: Creates a new tensor sharing data but NOT in the computation graph.
- **`torch.no_grad()`**: Context manager that disables gradient tracking entirely (saves memory during inference).

In [ ]:
# PyTorch Tensors & Autograd
import torch
import torch.nn.functional as F

# Tensor creation
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(f"Tensor: {x}")
print(f"Shape: {x.shape}, dtype: {x.dtype}, device: {x.device}")

# Autograd: track operations for automatic differentiation
w = torch.randn(3, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Forward pass: simple linear regression prediction
x0 = torch.tensor([1.0, 2.0, 3.0])
pred = x0 @ w + b
loss = (pred - 5.0)**2  # MSE vs target=5

# Backward pass: compute gradients
loss.backward()
print(f"\nAutograd demo:")
print(f"w.grad: {w.grad}")
print(f"b.grad: {b.grad}")
print(f"Gradient = d(loss)/dw = 2*(pred-5)*x0 = {(2*(pred-5.0)*x0).detach()}")

# Key: gradient flows BACKWARD through computational graph
print("\nComputational graph: x → @w → +b → pred → (pred-5)^2 → loss")

## Section 2 — `nn.Module` & Building Networks

### The `nn.Module` Contract
Every neural network in PyTorch is a subclass of `nn.Module`. The two things you **must** implement:
1. `__init__`: Define all layers and sub-modules. Call `super().__init__()` first.
2. `forward(x)`: Define the computation (what happens when you call `model(x)`).

PyTorch automatically handles: parameter registration, `.to(device)` propagation, `.parameters()` iteration, `state_dict` serialization.

### Parameter Registration
Anything assigned as `self.layer = nn.Linear(...)` (or any `nn.Module`) is **automatically registered** as a submodule. Its weights are included in `model.parameters()`.

Direct tensors are NOT registered unless you use `nn.Parameter(tensor)` or `self.register_buffer(name, tensor)` (for non-learnable state like BatchNorm running stats).

### Built-in Layers You Must Know
| Layer | Use case |
|-------|----------|
| `nn.Linear(in, out)` | Fully connected layer |
| `nn.Embedding(vocab, dim)` | Token/entity embeddings |
| `nn.Conv1d/Conv2d` | Sequence/image convolution |
| `nn.BatchNorm1d/2d` | Normalize activations |
| `nn.Dropout(p)` | Regularization (train only) |
| `nn.LayerNorm(dim)` | Normalize (transformer-style) |
| `nn.MultiheadAttention` | Self/cross-attention |
| `nn.Sequential` | Chain layers without subclassing |

In [ ]:
# Building Neural Networks with nn.Module
import torch
import torch.nn as nn
import torch.nn.functional as F

class ProteinClassifier(nn.Module):
    """Simple MLP for protein function classification."""
    def __init__(self, in_dim=20, hidden=64, n_classes=3, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden // 2)
        self.fc3 = nn.Linear(hidden // 2, n_classes)
        self.dropout = nn.Dropout(dropout)
        self.bn = nn.BatchNorm1d(hidden)

    def forward(self, x):
        x = F.relu(self.bn(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # logits
        return x

model = ProteinClassifier(in_dim=20, hidden=64, n_classes=3)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_params:,}")
print(f"Trainable: {trainable:,}")

# Test forward pass
batch = torch.randn(8, 20)  # 8 proteins, 20 amino acid features each
out = model(batch)
print(f"Input: {batch.shape} → Output logits: {out.shape}")
print(f"Predicted classes: {out.argmax(dim=1)}")

## Section 3 — Training Loop Fundamentals

### The Complete Training Loop
Every PyTorch training loop follows the same 5-step pattern per batch:

```
1. optimizer.zero_grad()   ← clear old gradients (CRITICAL: they accumulate by default)
2. outputs = model(inputs) ← forward pass
3. loss = criterion(outputs, targets)  ← compute loss
4. loss.backward()         ← backward pass: compute all gradients via autograd
5. optimizer.step()        ← update weights: w = w - lr * grad
```

### `torch.utils.data` — DataLoader
The `Dataset` + `DataLoader` pattern separates **data access** from **training logic**:
- `Dataset`: Define how to get one sample (implement `__len__` and `__getitem__`)
- `DataLoader`: Handles batching, shuffling, and multi-process loading

### Loss Functions
| Loss | Use case | Notes |
|------|----------|-------|
| `nn.MSELoss()` | Regression | Mean squared error |
| `nn.CrossEntropyLoss()` | Multi-class classification | Includes softmax internally |
| `nn.BCEWithLogitsLoss()` | Binary classification | More numerically stable than BCE+sigmoid |
| `nn.NLLLoss()` | With log_softmax output | Used in seq2seq |
| Custom | Any other objective | See Section 5 |

### Optimizers
| Optimizer | When to use |
|-----------|-------------|
| `torch.optim.SGD` | Computer vision, fine-tuning (with momentum) |
| `torch.optim.Adam` | Default choice for most tasks |
| `torch.optim.AdamW` | Transformers (Adam + decoupled weight decay) |
| `torch.optim.LBFGS` | Small models, second-order optimization |

In [ ]:
# Complete PyTorch Training Loop
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Synthetic data: gene expression → cancer type
torch.manual_seed(42)
n, feat, n_classes = 200, 20, 3
X = torch.randn(n, feat)
y = (X[:, 0] + X[:, 1]).long() % n_classes  # labels 0,1,2

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = nn.Sequential(
    nn.Linear(feat, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, n_classes)
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(20):
    model.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    losses.append(epoch_loss / n)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}: loss={losses[-1]:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    logits = model(X)
    acc = (logits.argmax(1) == y).float().mean()
print(f"\nFinal train accuracy: {acc:.3f}")

## Section 4 — Key PyTorch Patterns

### The Essential Toolkit
These 6 patterns appear in virtually every serious PyTorch project. They're not advanced — they're standard:

1. **`torch.no_grad()`** — Inference without gradient overhead
2. **`model.state_dict()`** — Extract all learnable weights as an ordered dict
3. **Checkpoint save/load** — Persist and resume training
4. **Device-agnostic code** — Works on CPU and any GPU without if/else branching
5. **Custom `Dataset`** — Connect any data source to PyTorch's DataLoader
6. **`model.train()` / `model.eval()`** — Switch between training and inference behavior

### Checkpoint Best Practices
Always save MORE than just the weights:
```python
{'epoch': epoch, 'model_state_dict': ..., 'optimizer_state_dict': ..., 'loss': best_loss}
```
Why? The optimizer has its own state (Adam momentum, variance estimates). Without it, resuming training gives different results.

In [ ]:
# DataLoader and Custom Datasets
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np

class DNADataset(Dataset):
    """Custom dataset for DNA sequences with one-hot encoding."""
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
        self.base_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # One-hot encode
        enc = torch.zeros(len(seq), 4)
        for i, b in enumerate(seq):
            if b in self.base_idx:
                enc[i, self.base_idx[b]] = 1.0
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return enc, label

# Generate synthetic sequences
np.random.seed(42)
seqs = [''.join(np.random.choice(list('ACGT'), 50)) for _ in range(100)]
labels = np.random.randint(0, 2, 100)

dataset = DNADataset(seqs, labels)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [train_size, len(dataset)-train_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)

print(f"Dataset: {len(dataset)} sequences")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# Check one batch
batch_x, batch_y = next(iter(train_loader))
print(f"Batch input shape: {batch_x.shape}")  # (16, 50, 4)
print(f"Batch labels: {batch_y}")

## Section 5 — Advanced PyTorch

### Custom Loss Functions
Any Python callable that takes `(predictions, targets)` and returns a scalar tensor can be a loss. The key: ALL operations must be PyTorch operations so gradients flow back.

### Hooks
Hooks let you intercept intermediate activations and gradients without modifying the model. Two types:
- `register_forward_hook(fn)`: Called after each forward pass through a layer
- `register_backward_hook(fn)`: Called after the backward pass through a layer

Use cases: feature visualization, gradient debugging, activation statistics.

### Mixed Precision (`torch.autocast`)
`torch.autocast` automatically casts operations to float16 where safe (matrix multiplications, convolutions) while keeping float32 where numerically required (softmax, loss computation). Combined with `GradScaler`, it prevents gradient underflow.

```
With mixed precision: ~2x speedup, ~2x memory reduction on NVIDIA Ampere+ GPUs
ESM2, AlphaFold3 training all use mixed precision.
```

### `requires_grad` Management
Fine-tuning pattern: freeze the pretrained backbone, train only the classification head.
```python
for param in model.backbone.parameters():
    param.requires_grad = False  # Frozen
for param in model.head.parameters():
    param.requires_grad = True   # Trainable
```

In [ ]:
# PyTorch: complete mini training example
import torch, torch.nn as nn

torch.manual_seed(0)
X = torch.randn(100, 5)
y = (X.sum(1) > 0).long()

model = nn.Sequential(nn.Linear(5, 16), nn.ReLU(), nn.Linear(16, 2))
opt = torch.optim.Adam(model.parameters())

for epoch in range(30):
    logits = model(X)
    loss = nn.CrossEntropyLoss()(logits, y)
    opt.zero_grad(); loss.backward(); opt.step()

acc = (model(X).argmax(1) == y).float().mean()
print(f"Accuracy after 30 epochs: {acc:.2f}")

## Section 6 — PyTorch for Sequences (Preview of Modules 05–07–10)

This section introduces the PyTorch building blocks used by protein language models (ESM2), AlphaFold3, and the fine-tuning capstone.

### `nn.Embedding` — Discrete Token → Continuous Vector
Maps integer indices (amino acid IDs, nucleotide IDs) to dense vectors. Internally: a lookup table of shape `(vocab_size, embedding_dim)`. Learnable.

### `nn.MultiheadAttention` — The Core of Transformers
Attention computes a **weighted sum** of values, where weights are determined by query-key similarity. Multi-head attention runs `h` attention heads in parallel, then concatenates:
```
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
```

### Positional Encoding
Attention is **permutation-invariant** (it doesn't know order). Positional encoding injects position information by adding a position-dependent signal to each token's embedding.

### Why This Matters
- **ESM2**: 650M-param protein language model. Architecture: Embedding + 33 transformer blocks each using MultiheadAttention + LayerNorm + FFN.
- **AlphaFold3**: Uses attention over residue pairs + single representation. The Evoformer is a stack of triangle attention + MultiheadAttention blocks.
- **Module 10 (Fine-tuning)**: You'll add a classification head on top of ESM2's CLS token embedding.

In [ ]:
# PyTorch: complete mini training example
import torch, torch.nn as nn

torch.manual_seed(0)
X = torch.randn(100, 5)
y = (X.sum(1) > 0).long()

model = nn.Sequential(nn.Linear(5, 16), nn.ReLU(), nn.Linear(16, 2))
opt = torch.optim.Adam(model.parameters())

for epoch in range(30):
    logits = model(X)
    loss = nn.CrossEntropyLoss()(logits, y)
    opt.zero_grad(); loss.backward(); opt.step()

acc = (model(X).argmax(1) == y).float().mean()
print(f"Accuracy after 30 epochs: {acc:.2f}")

## Section 7 — Debugging PyTorch

### The 5 Most Common Errors

| Error | Most Likely Cause | Fix |
|-------|-------------------|-----|
| `RuntimeError: Expected all tensors on the same device` | Mixed CPU/GPU tensors | `.to(device)` on inputs before forward pass |
| `RuntimeError: mat1 and mat2 shapes cannot be multiplied` | Shape mismatch in Linear | `print(x.shape)` before the failing layer |
| `RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation` | In-place op on tensor in graph | Use `x = x + 1` not `x += 1` |
| Loss is `nan` from step 1 | NaN in input data or exploding init | Check data, use `torch.isnan(x).any()` |
| Loss stuck / not decreasing | Forgot `zero_grad()`, `model.train()`, or wrong target shape | Step through training loop manually |

### Gradient Flow Debugging
A healthy gradient flow has all gradients non-zero and not exploding. Common issues:
- **Vanishing gradients**: Gradients become tiny in early layers (often in deep ReLU nets)
- **Exploding gradients**: Gradients grow exponentially (use gradient clipping)
- **Dead neurons**: ReLU neurons that always output 0 (use LeakyReLU or GELU)

In [ ]:
# Complete PyTorch Training Loop
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Synthetic data: gene expression → cancer type
torch.manual_seed(42)
n, feat, n_classes = 200, 20, 3
X = torch.randn(n, feat)
y = (X[:, 0] + X[:, 1]).long() % n_classes  # labels 0,1,2

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = nn.Sequential(
    nn.Linear(feat, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, n_classes)
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(20):
    model.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    losses.append(epoch_loss / n)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}: loss={losses[-1]:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    logits = model(X)
    acc = (logits.argmax(1) == y).float().mean()
print(f"\nFinal train accuracy: {acc:.3f}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- Automatic differentiation: reverse-mode AD, Wengert tape (computation trace), chain rule as graph traversal
- Optimization theory: SGD convergence proofs, Adam adaptive moments, learning rate schedules (cosine, warmup)
- Computational graph semantics: static (define-then-run) vs dynamic (define-by-run / eager)
- Key paper: Baydin et al. (2018) "Automatic Differentiation in Machine Learning: a Survey"

### 2️⃣ Must-Have Popular Resources
#### 🟢 Start Here (Zero PyTorch Background)
- 🆓 **Neural Networks: Zero to Hero (Karpathy)** — [youtube.com/playlist](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) — 8 videos, implement backprop from scratch
- 🆓 **fast.ai Part 1** — [course.fast.ai](https://course.fast.ai/) — free, top-down, learn by building in PyTorch
- 🆓 **PyTorch 60-Minute Blitz** — [pytorch.org/tutorials/beginner/blitz](https://pytorch.org/tutorials/beginner/blitz/blitz_intro.html) — official quick start
- 🆓 **MIT 6.S191 Deep Learning** — [introtodeeplearning.com](http://introtodeeplearning.com/) — free annual course, lab assignments
- 📘 Book: *Deep Learning with PyTorch* (Stevens et al.) — free official book at pytorch.org/assets
- 🎓 Course: Andrej Karpathy "Neural Networks: Zero to Hero" — [YouTube playlist](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) 4M+ views
- 🎓 Course: fast.ai Part 1 — top-down practical deep learning, free
- ⭐ GitHub: [pytorch/pytorch](https://github.com/pytorch/pytorch) — 85k★
- 📊 Kaggle: Intro to Deep Learning course (Kaggle Learn) — Keras-flavoured but concepts transfer

### 3️⃣ Practicals — Hands-On
- 💻 Exercise: PyTorch 60-minute blitz — official tutorial, covers tensors → autograd → CNN
- 💻 Exercise: Implement backprop from scratch with [karpathy/micrograd](https://github.com/karpathy/micrograd) — 9k★, ~150 lines
- 🔗 GitHub: [pyg-team/pytorch_geometric](https://github.com/pyg-team/pytorch_geometric) — 21k★, GNN intro for molecular graphs
- 🤗 HuggingFace: [facebook/esm2_t6_8M_UR50D](https://huggingface.co/facebook/esm2_t6_8M_UR50D) — run protein inference in 10 lines
- 📊 Kaggle: MNIST digit recognition — baseline CNN, then improve with BatchNorm/Dropout

### 4️⃣ Real-World Problems
- 🏭 Industry: ESM2/ESMFold (Meta), AlphaFold2/3 (Google structural biology research labs), Boltz-2 — all written in PyTorch
- 📊 Dataset: [UniRef50](https://huggingface.co/datasets/nferruz/UR50_2021_04) on HuggingFace — 50M protein sequences for LM pre-training
- 🔬 Application: Protein language model fine-tuning — ESM2 + linear head for stability/function prediction

### 5️⃣ Interview Tips
- ❓ Must know: why `.zero_grad()` before `.backward()` — PyTorch accumulates gradients by default (useful for RNNs, dangerous if forgotten)
- ❓ Must know: `model.train()` vs `model.eval()` — affects BatchNorm running stats and Dropout masking
- ⚠️ Common mistake: `DataParallel` vs `DistributedDataParallel` — DP has GIL bottleneck, DDP is the production choice
- ⚠️ Common mistake: not moving both model and data to the same device — silent CPU fallback
- 💡 Pro tip: `torch.compile()` in PyTorch 2.x gives 1.5–2x speedup with one line — mention this in interviews

### 6️⃣ Hot Industry Topics
- 🔥 Trending: `torch.compile` + Dynamo compiler — [pytorch.org/tutorials](https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html)
- 🔥 Trending: FlashAttention-2 — [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) 14k★, 2-4x faster attention
- 🔥 Trending: PyTorch 2.x ecosystem: `torch.export`, AOT Inductor for deployment
- 🚀 Build this: Fine-tune ESM2 (8M params) on a protein thermostability dataset — full PyTorch training loop with mixed precision (`torch.cuda.amp`)

## Interview Q&A — PyTorch Fundamentals

**Q: What is a computation graph and why does PyTorch use dynamic graphs?**
A: A computation graph records operations for automatic differentiation. TensorFlow (v1) used static graphs (define-then-run). PyTorch uses dynamic graphs (define-by-run) — you write normal Python code and PyTorch builds the graph on-the-fly. This makes debugging with `print()` and `pdb` work naturally.

**Q: What does `.detach()` do and when should you use it?**
A: `.detach()` creates a tensor that shares data but is excluded from the computation graph. Use when: (1) computing metrics (no gradient needed), (2) using a tensor as input but not wanting gradients to flow through it (e.g., target network in DQN), (3) converting to NumPy: `tensor.detach().numpy()`.

**Q: Explain `model.train()` vs `model.eval()` — what changes?**
A: `model.train()` enables Dropout (randomly zeros activations) and BatchNorm (uses batch statistics). `model.eval()` disables Dropout (all neurons active) and switches BatchNorm to use running statistics. Always call `model.eval()` before inference and `model.train()` before training.

**Q: What is gradient accumulation and why is it used?**
A: Instead of `optimizer.step()` every batch, accumulate gradients over N batches before stepping. Simulates a batch size N× larger without needing N× more GPU memory. Used when training large protein language models on limited hardware.

**Q: When would you use `torch.no_grad()` vs `torch.inference_mode()`?**
A: Both disable gradient computation. `inference_mode()` is stronger — it also disables version counting, making it ~10% faster. Use `inference_mode()` for validation/inference, `no_grad()` when you might need to access `.grad_fn` even without gradients.

### PyTorch Time Complexity (Operations)
| Operation | Time | Memory | Notes |
|-----------|------|--------|-------|
| `torch.mm(A, B)` where A:(n,k), B:(k,m) | O(n·k·m) | O(n·m) | Uses cuBLAS on GPU |
| Self-attention | O(L²·d) | O(L²) | L=seq length, d=dim |
| Convolution (1D, k kernel) | O(n·c_in·c_out·k) | O(n·c_out) | — |
| Backward pass | ~2-3× forward | same as forward | rule of thumb |
| DataLoader (num_workers=4) | O(batch/workers) | O(batch) | parallel prefetch |

In [ ]:
# PyTorch Tensors & Autograd
import torch
import torch.nn.functional as F

# Tensor creation
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(f"Tensor: {x}")
print(f"Shape: {x.shape}, dtype: {x.dtype}, device: {x.device}")

# Autograd: track operations for automatic differentiation
w = torch.randn(3, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Forward pass: simple linear regression prediction
x0 = torch.tensor([1.0, 2.0, 3.0])
pred = x0 @ w + b
loss = (pred - 5.0)**2  # MSE vs target=5

# Backward pass: compute gradients
loss.backward()
print(f"\nAutograd demo:")
print(f"w.grad: {w.grad}")
print(f"b.grad: {b.grad}")
print(f"Gradient = d(loss)/dw = 2*(pred-5)*x0 = {(2*(pred-5.0)*x0).detach()}")

# Key: gradient flows BACKWARD through computational graph
print("\nComputational graph: x → @w → +b → pred → (pred-5)^2 → loss")

## Cheat Sheet — Most-Used PyTorch Patterns

One-liners and short snippets for daily use. Memorize these.

```python
# ── Device ──────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = x.to(device)          # Move tensor
model = model.to(device)  # Move model (moves all parameters)

# ── Tensor creation ──────────────────────────────────────────
torch.zeros(B, L, D)       # All zeros  (batch, length, dim)
torch.ones_like(x)         # Same shape/device/dtype as x, all ones
torch.randn_like(x)        # Same shape/device/dtype as x, normal random
torch.arange(0, N, step)   # Like np.arange
torch.linspace(0, 1, N)    # N evenly spaced points in [0, 1]

# ── Shape ───────────────────────────────────────────────────
x.shape                    # torch.Size — tuple of ints
x.reshape(B, -1)           # -1 means 'infer this dim'
x.permute(2, 0, 1)         # Transpose arbitrary dims
x.unsqueeze(0)             # Add dim at position 0
x.squeeze(0)               # Remove dim of size 1 at position 0
torch.cat([a, b], dim=0)   # Concatenate along dim
torch.stack([a, b], dim=0) # Stack (adds new dimension)

# ── Training loop skeleton ───────────────────────────────────
model.train()
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Optional
    optimizer.step()

# ── Inference skeleton ───────────────────────────────────────
model.eval()
with torch.no_grad():
    preds = model(X.to(device))

# ── Checkpoint ──────────────────────────────────────────────
# Save
torch.save({'epoch': e, 'model': model.state_dict(), 'opt': opt.state_dict()}, 'ckpt.pt')
# Load
ckpt = torch.load('ckpt.pt', map_location=device)
model.load_state_dict(ckpt['model'])

# ── Parameter counts ─────────────────────────────────────────
total   = sum(p.numel() for p in model.parameters())
train   = sum(p.numel() for p in model.parameters() if p.requires_grad)

# ── Freeze / unfreeze ────────────────────────────────────────
for p in model.backbone.parameters(): p.requires_grad = False  # Freeze
for p in model.head.parameters():     p.requires_grad = True   # Unfreeze

# ── NaN check ────────────────────────────────────────────────
assert not torch.isnan(loss), 'Loss is NaN!'
assert not torch.isnan(x).any(), f'NaN in input at step {step}'

# ── Mixed precision ──────────────────────────────────────────
scaler = torch.cuda.amp.GradScaler()
with torch.autocast(device_type='cuda'):
    loss = criterion(model(X), y)
scaler.scale(loss).backward()
scaler.step(optimizer); scaler.update()

# ── Reproducibility ─────────────────────────────────────────
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
```

In [ ]:
# Complete PyTorch Training Loop
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Synthetic data: gene expression → cancer type
torch.manual_seed(42)
n, feat, n_classes = 200, 20, 3
X = torch.randn(n, feat)
y = (X[:, 0] + X[:, 1]).long() % n_classes  # labels 0,1,2

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = nn.Sequential(
    nn.Linear(feat, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, n_classes)
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(20):
    model.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    losses.append(epoch_loss / n)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}: loss={losses[-1]:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    logits = model(X)
    acc = (logits.argmax(1) == y).float().mean()
print(f"\nFinal train accuracy: {acc:.3f}")

## Mastery Check

Before moving on, make sure you can:
1. explain the core concept of this notebook in plain English without looking
2. write a small toy example from scratch
3. identify one common mistake and explain why it is wrong
4. say whether you should revisit math, Python, or ML basics before continuing


---
## 📚 Resources — PyTorch & Deep Learning Foundations

### University Courses (All Free)
| Course | Institution | Why This One |
|--------|-------------|-------------|
| **[MIT 6.S191 Deep Learning](https://introtodeeplearning.com/)** | MIT | Best video lectures for PyTorch; Lab 1 covers this notebook exactly |
| **[Stanford CS231n](https://cs231n.github.io/)** | Stanford | Notes + assignments on neural nets, backprop, optimization |
| **[fast.ai Practical Deep Learning](https://course.fast.ai/)** | fast.ai | Best for coding-first learners; free; covers PyTorch thoroughly |
| **[Deep Learning with PyTorch: Zero to GANs](https://jovian.com/learn/deep-learning-with-pytorch-zero-to-gans)** | Jovian/freeCodeCamp | Free video + Jupyter; 6 weeks |

### Essential Reading (Free)
- **[Deep Learning Book](https://www.deeplearningbook.org/)** (Goodfellow et al. 2016) — Chapters 6-8: feedforward nets, regularization, optimization. Free PDF.
- **[PyTorch Official Tutorials](https://pytorch.org/tutorials/)** — `Learn the Basics` sequence (6 tutorials, ~4 hours)
- **[Andrej Karpathy: Neural Networks — Zero to Hero](https://karpathy.ai/zero-to-hero.html)** — builds backprop from scratch; essential for deep understanding

### The 3 Things to Master in PyTorch
1. **Autograd**: `requires_grad=True`, `.backward()`, `.grad` — understand what gets computed
2. **DataLoader**: `Dataset`, `DataLoader`, `collate_fn` — how data flows to GPU
3. **Training loop**: `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` — the ritual

### MIT OCW Relevant Lectures
- **[MIT 18.065 Matrix Methods in Data Analysis](https://ocw.mit.edu/courses/18-065-matrix-methods-in-data-analysis-signal-processing-and-machine-learning-spring-2018/)** — Gilbert Strang on linear algebra for ML. Lecture 7: backpropagation as chain rule.
- **[MIT 6.036 Intro to ML](https://introml.mit.edu/spring22)** — free notes; Module 6 covers neural networks from scratch

### When You Get Stuck
- **[PyTorch Discuss Forum](https://discuss.pytorch.org/)** — official forum, usually answered within hours
- **[Yannic Kilcher YouTube](https://www.youtube.com/@YannicKilcher)** — paper implementations explained
